# Eval: OpenAI **Batch API** (−50% cost)

Notebook **riêng cho OpenAI official** (`api.openai.com`) — **không** dùng OpenRouter.

Pipeline:
1. **Build** JSONL requests (cùng prompt / MODE với realtime)
2. **Submit** batch (`completion_window=24h`)
3. **Poll** status
4. **Ingest** → `results/<model>/<MODE>/{calls,scores.jsonl,metrics.json}` (dùng chung với `20_compare_summary.ipynb`)

Trade-off: rẻ hơn ~50%, nhưng có thể mất tới 24h (thường nhanh hơn).

Cần `OPENAI_API_KEY` (trong `doan/.env` hoặc gán `TOKEN`).

In [36]:
# === INPUTS ===
MODEL = "gpt-5.6-sol"  # model OpenAI official (vd. gpt-4.1-mini, gpt-5.6-sol)
TOKEN = ""  # trống → OPENAI_API_KEY từ doan/.env
MODE = "T"  # ORIG | S | T | ST | ST-E

N_SAMPLES = 10
# GPT-5.x thường chỉ cho temperature mặc định (=1). Giá trị 1.5 sẽ bị omit trên GPT-5.
TEMPERATURE = 1  # paper closed; với GPT-5 builder omit nếu ≠ 1
REASONING_EFFORT = "medium"  # chỉ khi MODE có thinking (T/ST/ST-E)

# Giới hạn câu để smoke (None = full 50)
LIMIT_SENTENCES = None

# Poll
POLL_SEC = 30
WAIT_FOR_COMPLETION = True  # False = submit rồi dừng; chạy lại cell poll/ingest sau


In [37]:
%pip install -q openai pyyaml tqdm
%pip install -q -e ../..



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [38]:
import os, sys
from pathlib import Path

HERE = Path.cwd().resolve()
REPO = None
for c in [HERE, *HERE.parents]:
    if (c / "configs" / "experiment.yaml").exists():
        REPO = c
        break
    if (c / "doan" / "configs" / "experiment.yaml").exists():
        REPO = c / "doan"
        break
assert REPO is not None, "Chạy notebook từ trong repo doan/"
sys.path.insert(0, str(REPO / "src"))

env_file = REPO / ".env"
if env_file.exists():
    for line in env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k, v = k.strip(), v.strip().strip('"').strip("'")
        if k and k not in os.environ:
            os.environ[k] = v
    print("Loaded .env:", env_file)

if not TOKEN:
    TOKEN = os.environ.get("OPENAI_API_KEY") or ""
if not TOKEN:
    raise RuntimeError("Cần OPENAI_API_KEY (official OpenAI) — không dùng OpenRouter key")

print("REPO:", REPO)
print("MODEL:", MODEL)
print("MODE:", MODE)
print("N_SAMPLES:", N_SAMPLES)
print("TEMPERATURE:", TEMPERATURE)
print("TOKEN set:", bool(TOKEN), "(len=", len(TOKEN), ")")


Loaded .env: /Users/nguyenkz/Documents/code/CS2202.CH202/doan/.env
REPO: /Users/nguyenkz/Documents/code/CS2202.CH202/doan
MODEL: gpt-5.6-sol
MODE: T
N_SAMPLES: 10
TEMPERATURE: 1
TOKEN set: True (len= 164 )


## 1) Build JSONL

Resume: bỏ câu đã đủ `n_samples` trong `scores.jsonl` và call file đã có.

In [39]:
from plausibility_eval.openai_batch import build_batch_requests

built = build_batch_requests(
    model=MODEL,
    mode=MODE,
    repo=REPO,
    n_samples_override=N_SAMPLES,
    limit_sentences=LIMIT_SENTENCES,
    temperature_override=TEMPERATURE,
    reasoning_effort=REASONING_EFFORT,
    resume=True,
)
print("n_requests:", built["n_requests"])
print("requests:", built["requests_path"])
print("batch_dir:", built["batch_dir"])
print("meta:", built["meta"])
assert built["n_requests"] > 0, "Không còn request nào — có thể đã đủ scores/calls"


n_requests: 300
requests: /Users/nguyenkz/Documents/code/CS2202.CH202/doan/results/gpt-5.6-sol/T/batch/requests.jsonl
batch_dir: /Users/nguyenkz/Documents/code/CS2202.CH202/doan/results/gpt-5.6-sol/T/batch
meta: {'created_at': '2026-07-26T19:25:38+07:00', 'model': 'gpt-5.6-sol', 'mode': 'T', 'flags': {'schema': False, 'thinking': True, 'examples': True}, 'n_samples': 10, 'temperature': 1.0, 'max_tokens': 1024, 'reasoning_effort': 'medium', 'n_requests': 300, 'n_sentences': 50, 'skipped_complete_sentences': 0, 'skipped_existing_calls': 0, 'requests_path': 'results/gpt-5.6-sol/T/batch/requests.jsonl', 'protocol': 'openai_batch_50pct'}


## 2) Submit batch

In [40]:
from plausibility_eval.openai_batch import submit_batch

job = submit_batch(
    requests_path=built["requests_path"],
    token=TOKEN,
    batch_dir=built["batch_dir"],
)
BATCH_ID = job["batch_id"]
print("batch_id:", BATCH_ID)
print("status:", job["status"])
print("saved:", built["batch_dir"] / "batch_job.json")


batch_id: batch_6a65fcc407b481908bba12067687560f
status: validating
saved: /Users/nguyenkz/Documents/code/CS2202.CH202/doan/results/gpt-5.6-sol/T/batch/batch_job.json


## 3) Poll status

Có thể tắt `WAIT_FOR_COMPLETION` và quay lại cell này sau.

In [41]:
from pathlib import Path
import json
from plausibility_eval.openai_batch import get_batch_status, wait_batch

batch_dir = built["batch_dir"] if "built" in dir() else None
if batch_dir is None:
    # recover from disk
    from plausibility_eval.io_utils import results_dir
    batch_dir = results_dir(REPO, MODEL, MODE) / "batch"
    job_path = batch_dir / "batch_job.json"
    assert job_path.exists(), f"Thiếu {job_path} — chạy submit trước"
    BATCH_ID = json.loads(job_path.read_text())["batch_id"]

if WAIT_FOR_COMPLETION:
    status = wait_batch(
        batch_id=BATCH_ID,
        token=TOKEN,
        batch_dir=batch_dir,
        poll_sec=POLL_SEC,
    )
else:
    status = get_batch_status(batch_id=BATCH_ID, token=TOKEN, batch_dir=batch_dir)

print("status:", status["status"])
print("output_file_id:", status.get("output_file_id"))
print("error_file_id:", status.get("error_file_id"))
print("counts:", status.get("request_counts"))


[batch] status=validating counts={'completed': 0, 'failed': 0, 'total': 0}
[batch] status=in_progress counts={'completed': 0, 'failed': 0, 'total': 300}
[batch] status=in_progress counts={'completed': 0, 'failed': 0, 'total': 300}
[batch] status=in_progress counts={'completed': 26, 'failed': 0, 'total': 300}
[batch] status=in_progress counts={'completed': 86, 'failed': 0, 'total': 300}
[batch] status=finalizing counts={'completed': 300, 'failed': 0, 'total': 300}
[batch] status=completed counts={'completed': 300, 'failed': 0, 'total': 300}
status: completed
output_file_id: file-5iEKZGpChQW1qPMtaqpBuy
error_file_id: None
counts: {'completed': 300, 'failed': 0, 'total': 300}


## 4) Download + ingest vào `results/`

In [42]:
from plausibility_eval.openai_batch import download_batch_files, ingest_batch_results
from plausibility_eval.io_utils import results_dir
import json

batch_dir = results_dir(REPO, MODEL, MODE) / "batch"
job = json.loads((batch_dir / "batch_job.json").read_text())
assert job.get("status") == "completed", f"Batch chưa completed: {job.get('status')}"

paths = download_batch_files(
    token=TOKEN,
    batch_dir=batch_dir,
    output_file_id=job.get("output_file_id"),
    error_file_id=job.get("error_file_id"),
)
print("downloaded:", paths)

result = ingest_batch_results(
    model=MODEL,
    mode=MODE,
    repo=REPO,
    results_path=paths["results_path"],
    errors_path=paths.get("errors_path"),
    n_samples_override=N_SAMPLES,
)
print("out_dir:", result["out_dir"])
print("saved_calls:", result["saved_calls"], "failed_lines:", result["failed_lines"])
print("n_scores:", result["n_scores"])
print("metrics:", result["metrics"])


downloaded: {'results_path': PosixPath('/Users/nguyenkz/Documents/code/CS2202.CH202/doan/results/gpt-5.6-sol/T/batch/results.jsonl'), 'errors_path': None}
out_dir: /Users/nguyenkz/Documents/code/CS2202.CH202/doan/results/gpt-5.6-sol/T
saved_calls: 300 failed_lines: 0
n_scores: 50
metrics: {'n_sentences': 50, 'n_scored': 50, 'pearson_r': 0.7061714127966138, 'mae': 0.6757639999999999, 'rmse': 0.903664979513979, 'parse_fail_count': 0, 'parse_fail_rate': 0.0, 'usage_totals': {'input_tokens': 338580, 'output_tokens': 28933, 'reasoning_tokens': 11961, 'total_tokens': 379474, 'n_api_calls': 500}, 'mean_tokens_per_sentence': {'input': 6771.6, 'output': 578.66, 'reasoning': 239.22}, 'latency_ms_total': 0, 'model_id': 'gpt-5.6-sol', 'mode': 'T', 'provider': 'openai_official_batch'}


## Gợi ý

- Smoke: `LIMIT_SENTENCES = 5`, `N_SAMPLES = 2` rồi full.
- MODE thinking (`T`/`ST`): model phải hỗ trợ `reasoning_effort` (vd. GPT-5.x).
- Summary cost: chạy `20_compare_summary.ipynb` — nhớ batch thường **½ giá** realtime trong `pricing.yaml` nếu bạn tách dòng giá batch.
- Artifact trung gian: `results/<model>/<MODE>/batch/{requests,results,batch_job}.jsonl|json`